In [38]:
from qdrant_client import QdrantClient   
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai

## Read the sampled dataset with Amazon inventory data

In [39]:
df_items = pd.read_json(
    "/Users/benny/AI-Engineer/data/meta_Books_fiction_rating_filtered_subset.jsonl", lines=True
)

In [40]:
df_items.head()

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Books,Conqueror's Kiss,"Mass Market Paperback – November 1, 2006",{'avatar': 'https://m.media-amazon.com/images/...,4.4,450,[There is little hope for sanctuary in the mid...,[],1.35,[{'large': 'https://m.media-amazon.com/images/...,[],Hannah Howell (Author),"[Books, Literature & Fiction, Genre Fiction]","{'Publisher': 'Zebra (November 1, 2006)', 'Lan...",0821780050,NaN
1,Books,Rebel Hunter (CERBERUS),"Paperback – October 26, 2020",{'avatar': 'https://m.media-amazon.com/images/...,4.7,351,[Nolan Garrett has just declared a war on terr...,[],4.27,[{'large': 'https://m.media-amazon.com/images/...,[],Andy Peloquin (Author),"[Books, Science Fiction & Fantasy, Science Fic...",{'Publisher': 'Independently published (Octobe...,B08M2BKBNG,NaN
2,Buy a Kindle,Truth is the Soul of the Sun - A Biographical ...,NaN,{'avatar': 'https://m.media-amazon.com/images/...,3.9,32,[REVISED SECOND EDITIONThis is the story of Ha...,"[Review, Truth is the Soul of the Sun, Maria I...",0.0,[{'large': 'https://m.media-amazon.com/images/...,[],Maria Isabel Pita (Author) Format: Kindle Ed...,"[Books, Literature & Fiction, Genre Fiction]",{'Publisher': 'Lucid Press; 2nd edition (July ...,B002HWRUFW,NaN
3,Books,Iphigeneia at Aulis (Greek Tragedy in New Tran...,Reprint Edition,{'avatar': 'https://m.media-amazon.com/images/...,3.9,32,[Merwin and Dimock have provided a new transla...,"[Review, ""Wonderful edition.""―Professor Coni M...",9.18,[{'large': 'https://m.media-amazon.com/images/...,[],"Euripides (Author), W. S. Merwin (Translator)...","[Books, Literature & Fiction, History & Critic...",{'Publisher': 'Oxford University Press; Reprin...,0195077091,NaN
4,Books,The Rowan,"Hardcover – August 20, 1990",{'avatar': 'https://m.media-amazon.com/images/...,4.7,1368,"[The Rowan, a beautiful and powerful psychic w...","[From Publishers Weekly, McCaffrey leaves behi...",24.01,[{'large': 'https://m.media-amazon.com/images/...,[],Anne McCaffrey (Author),"[Books, Science Fiction & Fantasy, Science Fic...",{'Publisher': 'Putnam Adult; First Edition (Au...,0399135707,NaN


In [41]:
empty_descriptions = df_items["description"].str.len().eq(0)

print(empty_descriptions.sum())

504


In [42]:
list(df_items['features'].items())[0]

(0,
 ['There is little hope for sanctuary in the midst of the tumultuous Scottish border wars, yet one woman may find safe refuge...\xa0 in the arms of her sworn enemy...',
  'Jennet Graeme has witnessed terrible tragedy during the many years of strife between the Scots and the English. As Scottish invaders plunder her convent sanctuary, she defiantly resists the blond warrior who claims her as his prize. But his brute strength is overpowering and Jennet is forced to ride with him through the lawless lands, tending to the wounded, protected and desired by a man she wants to hate... but cannot...',
  "Sir Hacon Gillard is moved by Jennet's compassion and mercy. As a loyal knight, he's pledged fealty to his king's command, even as he loses his heart to this remarkable woman. Merciless in combat... yet there burns within him a spark for something far beyond the heat of battle..."])

In [43]:
empty_images = df_items["images"].str.len().eq(0)

print(empty_images.sum())

284


In [44]:
list(df_items['images'].items())[0:10]

[(0,
  [{'large': 'https://m.media-amazon.com/images/I/51QK4k8PimL._SY291_BO1,204,203,200_QL40_FMwebp_.jpg',
    'variant': 'MAIN'}]),
 (1,
  [{'large': 'https://m.media-amazon.com/images/I/51Pbtp7o4tL._SX326_BO1,204,203,200_.jpg',
    'variant': 'MAIN'}]),
 (2,
  [{'large': 'https://m.media-amazon.com/images/I/41HMu6eIDlL.jpg',
    'variant': 'MAIN'}]),
 (3,
  [{'large': 'https://m.media-amazon.com/images/I/41o2HIFBESL._SY291_BO1,204,203,200_QL40_FMwebp_.jpg',
    'variant': 'MAIN'}]),
 (4,
  [{'large': 'https://m.media-amazon.com/images/I/511yvDwZBPL._SX345_BO1,204,203,200_.jpg',
    'variant': 'MAIN'}]),
 (5,
  [{'large': 'https://m.media-amazon.com/images/I/518ZBPngBVL._SX326_BO1,204,203,200_.jpg',
    'variant': 'MAIN'}]),
 (6,
  [{'large': 'https://m.media-amazon.com/images/I/51RxWwTcdLL._SY291_BO1,204,203,200_QL40_FMwebp_.jpg',
    'variant': 'MAIN'}]),
 (7, []),
 (8, []),
 (9, [])]

#### Looks like there are a lot of empty description and pictures, we don't want those for the embedded data set

In [45]:
df_filtered = df_items[~empty_descriptions & ~empty_images]

In [46]:
len(df_filtered)

411

## Preprocess title and features

In [47]:
def preprocess_description(row):
    return f"{row['title']}{' '.join(row['features'])}"

In [48]:
def get_image_url(row):
    return row['images'][0].get('large')

In [49]:
df_filtered['description'] = df_filtered.apply(preprocess_description, axis=1)

In [65]:
df_filtered['image_url'] = df_filtered.apply(get_image_url, axis=1)

In [66]:
df_filtered.head()

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,image_urk,image_url
2,Buy a Kindle,Truth is the Soul of the Sun - A Biographical ...,NaN,{'avatar': 'https://m.media-amazon.com/images/...,3.9,32,[REVISED SECOND EDITIONThis is the story of Ha...,Truth is the Soul of the Sun - A Biographical ...,0.0,[{'large': 'https://m.media-amazon.com/images/...,[],Maria Isabel Pita (Author) Format: Kindle Ed...,"[Books, Literature & Fiction, Genre Fiction]",{'Publisher': 'Lucid Press; 2nd edition (July ...,B002HWRUFW,NaN,https://m.media-amazon.com/images/I/41HMu6eIDl...,https://m.media-amazon.com/images/I/41HMu6eIDl...
3,Books,Iphigeneia at Aulis (Greek Tragedy in New Tran...,Reprint Edition,{'avatar': 'https://m.media-amazon.com/images/...,3.9,32,[Merwin and Dimock have provided a new transla...,Iphigeneia at Aulis (Greek Tragedy in New Tran...,9.18,[{'large': 'https://m.media-amazon.com/images/...,[],"Euripides (Author), W. S. Merwin (Translator)...","[Books, Literature & Fiction, History & Critic...",{'Publisher': 'Oxford University Press; Reprin...,0195077091,NaN,https://m.media-amazon.com/images/I/41o2HIFBES...,https://m.media-amazon.com/images/I/41o2HIFBES...
4,Books,The Rowan,"Hardcover – August 20, 1990",{'avatar': 'https://m.media-amazon.com/images/...,4.7,1368,"[The Rowan, a beautiful and powerful psychic w...","The RowanThe Rowan, a beautiful and powerful p...",24.01,[{'large': 'https://m.media-amazon.com/images/...,[],Anne McCaffrey (Author),"[Books, Science Fiction & Fantasy, Science Fic...",{'Publisher': 'Putnam Adult; First Edition (Au...,0399135707,NaN,https://m.media-amazon.com/images/I/511yvDwZBP...,https://m.media-amazon.com/images/I/511yvDwZBP...
6,Books,Taras Bulba (Modern Library Classics),NaN,{'avatar': 'https://m.media-amazon.com/images/...,4.1,136,"[The First New Translation in Forty Years, Set...",Taras Bulba (Modern Library Classics)The First...,None,[{'large': 'https://m.media-amazon.com/images/...,[],"Nikolai Gogol (Author), Peter Constantine (Tr...","[Books, Literature & Fiction, Genre Fiction]","{'Publisher': 'Modern Library (December 30, 20...",0812971191,NaN,https://m.media-amazon.com/images/I/51RxWwTcdL...,https://m.media-amazon.com/images/I/51RxWwTcdL...
11,Books,Dr. Hugh Mann (Zombie Fallout),"Paperback – June 28, 2011",{'avatar': 'https://m.media-amazon.com/images/...,4.5,699,[Before there were zombies.....There was the v...,Dr. Hugh Mann (Zombie Fallout)Before there wer...,8.24,[{'large': 'https://m.media-amazon.com/images/...,[],Mark Tufo (Author),"[Books, Literature & Fiction, Genre Fiction]",{'Publisher': 'CreateSpace Independent Publish...,1463669364,NaN,https://m.media-amazon.com/images/I/51Q4591AJv...,https://m.media-amazon.com/images/I/51Q4591AJv...


#### Define the embedding function

In [54]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(
    "/Users/benny/AI-Engineer/.env",
    override=True,
)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

response = client.embeddings.create(
    input="random text",
    model="text-embedding-3-small",
)

In [56]:
print(client.base_url)

https://api.openai.com/v1/


In [57]:
len(response.data[0].embedding)

1536

In [59]:
def get_embedding(text, model="text-embedding-3-small"):
    response = client.embeddings.create(
        input=text,
        model=model,
    )
    return response.data[0].embedding

In [60]:
get_embedding('Benny')

[-0.0193634033203125,
 -0.057861328125,
 -0.05908203125,
 0.0182952880859375,
 -0.067138671875,
 -0.0222625732421875,
 -0.002685546875,
 0.0654296875,
 -0.0341796875,
 -0.035736083984375,
 -0.0007433891296386719,
 0.00365447998046875,
 0.037811279296875,
 -0.007488250732421875,
 0.00792694091796875,
 0.05230712890625,
 -0.038238525390625,
 0.03472900390625,
 -0.071533203125,
 -0.01517486572265625,
 -0.0155792236328125,
 0.01100921630859375,
 0.035247802734375,
 -0.0214385986328125,
 0.026885986328125,
 -0.0303192138671875,
 -0.001705169677734375,
 0.0135650634765625,
 0.03472900390625,
 -0.018585205078125,
 0.056793212890625,
 -0.0287628173828125,
 0.062103271484375,
 -0.0457763671875,
 -0.011505126953125,
 -0.0305023193359375,
 -0.0035419464111328125,
 -0.033172607421875,
 0.01381683349609375,
 -0.0263671875,
 -0.0548095703125,
 -0.005519866943359375,
 -0.0081024169921875,
 0.0201873779296875,
 0.031524658203125,
 -0.01464080810546875,
 -0.027984619140625,
 -0.0186767578125,
 0.040405

#### Create Qdrant Collection

In [83]:
qdrant_client = QdrantClient(
    url="http://localhost:6333")

In [84]:
qdrant_client.create_collection(
    collection_name="amazon_books_collection_sampled",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

True

#### Embed data

In [63]:
pointstruct = PointStruct(
    id=0,
    vector = get_embedding('test'),
    payload = {
        "title": "test",
        "model": "text-embedding-3-small",
    }, 
)

In [64]:
pointstruct

PointStruct(id=0, vector=[-0.00986480712890625, 0.0015592575073242188, 0.0156707763671875, -0.0548095703125, -0.00640869140625, -0.012908935546875, 0.00966644287109375, -0.01355743408203125, 0.0286712646484375, 0.007843017578125, 0.03179931640625, -0.0065765380859375, 0.0038814544677734375, 0.01012420654296875, 0.0140533447265625, 0.044677734375, -0.059783935546875, -0.002445220947265625, -0.0511474609375, 0.036529541015625, 0.036163330078125, 0.02435302734375, 0.03997802734375, -0.043609619140625, 0.034454345703125, -0.0195770263671875, -0.01401519775390625, 0.0106353759765625, 0.03118896484375, -0.041534423828125, 0.05877685546875, -0.02886962890625, -0.0013904571533203125, -0.038818359375, 0.055206298828125, 0.004138946533203125, 0.0236053466796875, 0.018890380859375, -0.0056915283203125, -0.003925323486328125, -0.041473388671875, -0.044921875, 0.0113677978515625, 0.0272064208984375, 0.0299835205078125, -0.0186767578125, -0.03436279296875, -0.038665771484375, 0.0484619140625, 0.0278

#### Embed all the amazon book data

In [67]:
data_to_embed = df_filtered[['description','image_url','rating_number','price','average_rating','parent_asin']].to_dict(orient='records') 

In [73]:
len(data_to_embed)

411

In [68]:
data_to_embed

[{'description': 'Truth is the Soul of the Sun - A Biographical Novel of Hatshepsut-MaatkareREVISED SECOND EDITIONThis is the story of Hatshepsut, the ancient Egyptian princess and queen who was crowned king, and for more than twenty years ruled on the Horus Throne of the Living as the Female Falcon. The name Hatshepsut chose for herself can be read as "Maat is the Ka of Re" which translated means "The True and Beautiful Manifestation of the Sun’s Divine Life-force." A more poetic but still accurate rendition is "Truth is the Soul of the Sun."Before becoming Pharaoh, Hatshepsut served as God’s Wife of Amun, an important economic and spiritual office created by her grandfather that holds the key to her mysterious and unprecedented power. Maatkare was not only a charismatic political leader, she was a mystic who inspired the wholehearted devotion of brilliant men. One of them was Senmut, a commoner she elevated to unparalleled heights of authority. But her most influential advocate was H

In [70]:
pointstructs = []

for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data['description'])
    pointstructs.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data,)
    )

In [71]:
pointstructs

[PointStruct(id=0, vector=[0.033477783203125, -0.0006651878356933594, 0.01898193359375, 0.044342041015625, -0.00020992755889892578, 0.00974273681640625, -0.0277557373046875, 0.0279083251953125, -0.035369873046875, 0.00801849365234375, 0.0236968994140625, -0.042327880859375, -0.0104217529296875, 0.00019788742065429688, 0.0301666259765625, 0.018096923828125, 0.002681732177734375, -0.0189056396484375, -0.00218963623046875, -0.0194854736328125, 0.0330810546875, -0.003513336181640625, 0.049896240234375, 0.0157623291015625, 0.0145263671875, 0.08740234375, -0.0277099609375, -0.0008172988891601562, 0.03692626953125, -0.0504150390625, -0.00121307373046875, 0.0006604194641113281, 0.018096923828125, -0.046600341796875, -0.0229949951171875, 0.0271148681640625, 0.041107177734375, -0.03668212890625, 0.0109100341796875, -0.0093231201171875, 0.045684814453125, -0.045501708984375, -0.0272979736328125, 0.022216796875, -0.0094146728515625, -0.0294952392578125, -0.0016021728515625, -0.026275634765625, 0.0

In [72]:
len(pointstructs)

411

#### Write embedded data to Qdrant

In [85]:
qdrant_client.upsert(
    collection_name="amazon_books_collection_sampled",
    wait=True,
    points=pointstructs,
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

#### Try retrieving

In [86]:
def retrieve_similar_books(query, top_k=5):
    query_embedding = get_embedding(query)
    search_result = qdrant_client.query_points(
        collection_name="amazon_books_collection_sampled",
        query=query_embedding,
        limit=top_k,
    )
    return search_result

In [87]:
retrieve_similar_books('Fiction books that are actually philosophy books in cover')

QueryResponse(points=[ScoredPoint(id=352, version=1, score=0.35829702, payload={'description': 'How People Change"At a time when slick, superficial, psychological works are foisted on the lay-public, Allen Wheelis has written a serious treatise."-- San Francisco Sunday Examiner-Chronicle', 'image_url': 'https://m.media-amazon.com/images/I/5177lREKDTL._SX326_BO1,204,203,200_.jpg', 'rating_number': 102, 'price': 10.59, 'average_rating': 4.2, 'parent_asin': '006090447X'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=156, version=1, score=0.3465468, payload={'description': 'Why I Read: The Serious Pleasure of Books"Wendy Lesser\'s extraordinary alertness, intelligence, and curiosity have made her one of America\'s most significant cultural critics," writes Stephen Greenblatt. In Why I Read , Lesser draws on a lifetime of pleasure reading and decades of editing one of the most distinguished literary magazines in the country, The Threepenny Review , to describe her love of 